In [ ]:
# --- 0. MOUNT DRIVE & IMPORTS ---
from google.colab import drive
drive.mount('/content/drive')

import os
import copy
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

print("\nLibraries imported and Drive mounted successfully!")


Libraries imported and Drive mounted successfully!


## 1. Setup & Paths
Define the project directories and select the compute device.

In [ ]:
# --- 1. SETUP & PATHS ---
PROJECT_FOLDER = r'/content/drive/MyDrive/Franklin_Project_2024'
ANALYSIS_FOLDER = f"{PROJECT_FOLDER}/data/analysis"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


## 2. Dataset Definition
Custom PyTorch `Dataset` to load images based on the master file generated in the previous step.

In [ ]:
# --- 2. DATASET DEFINITION ---
class CustomImageDataset(Dataset):
    def __init__(self, scenario, split_name, transform=None):
        self.transform = transform
        output_scenario_file = f"{ANALYSIS_FOLDER}/image_master_file_{scenario}_pedestrian.csv"

        df = pd.read_csv(output_scenario_file)
        df = df[df["split"] == split_name]

        # Convert text labels to binary integers (Assuming 'Positive'=1, 'Negative'=0)
        df['label_int'] = df['label'].map({'Positive': 1, 'Negative': 0})

        # Build full image paths
        # Adjust base path based on your folder structure (e.g., images/100k/train)
        base_img_dir = f"{PROJECT_FOLDER}/data/bdd100k/bdd100k/images/100k/{split_name}"
        df["full_path"] = base_img_dir + "/" + df["image_name"]

        self.image_paths = df[["full_path", "label_int"]].values

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path, label = self.image_paths[idx]
        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        label = torch.tensor(label, dtype=torch.long)
        return image, label

## 3. Normalization & Dataloaders
Toggle between Conventional (ImageNet) and Custom (Calculated BDD100k stats) normalization.

In [ ]:
# --- 3. NORMALIZATION TOGGLE ---
def get_transforms(scenario, norm_type='custom'):
    """Toggles between Conventional (ImageNet) and Custom (Your calculated BDD100k stats)"""
    if norm_type == 'conventional':
        mean, std = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
    else:
        # Plug in the exact stats calculated from your 02_RGB_Analysis notebook
        stats = {
            'city': {'mean': [0.274, 0.280, 0.276], 'std': [0.243, 0.256, 0.264]},
            'highway': {'mean': [0.277, 0.305, 0.304], 'std': [0.253, 0.283, 0.299]},
            'residential': {'mean': [0.306, 0.331, 0.331], 'std': [0.250, 0.266, 0.277]},
            'overall': {'mean': [0.285, 0.305, 0.303], 'std': [0.249, 0.268, 0.280]}
        }
        mean, std = stats.get(scenario, stats['overall'])['mean'], stats.get(scenario, stats['overall'])['std']

    return transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=mean, std=std)
    ])

def get_dataloaders(scenario, norm_type='custom', batch_size=32):
    transform = get_transforms(scenario, norm_type)
    dataloaders = {
        'train': DataLoader(CustomImageDataset(scenario, 'train', transform), batch_size=batch_size, shuffle=True, num_workers=2),
        'val': DataLoader(CustomImageDataset(scenario, 'val', transform), batch_size=batch_size, shuffle=False, num_workers=2),
        'test': DataLoader(CustomImageDataset(scenario, 'test', transform), batch_size=batch_size, shuffle=False, num_workers=2)
    }
    return dataloaders

## 4. Model Architectures
Defining the Custom CNN and configuring VGG16 with frozen layers.

In [ ]:
# --- 4. MODEL ARCHITECTURES ---
class CustomCNN(nn.Module):
    def __init__(self, num_classes=2):
        super(CustomCNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2, 2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2, 2)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 28 * 28, 512), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        return self.classifier(self.features(x))

def get_vgg16(num_classes=2):
    model = models.vgg16(weights=models.VGG16_Weights.DEFAULT)
    # Freeze first 15 layers as mentioned in your methodology
    for i, param in enumerate(model.features.parameters()):
        if i < 15:
            param.requires_grad = False

    num_features = model.classifier[6].in_features
    model.classifier[6] = nn.Linear(num_features, num_classes)
    return model

## 5. Training Loop & Evaluation Functions
Core functions to train the model, track history for plots, and evaluate accuracy/confusion matrix.

In [ ]:
# --- 5. TRAINING LOOP (WITH HISTORY FOR OVERFITTING PLOTS) ---
def train_model(model, dataloaders, criterion, optimizer, num_epochs=10):
    model = model.to(device)
    best_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

    for epoch in range(num_epochs):
        print(f'Epoch {epoch+1}/{num_epochs}')
        for phase in ['train', 'val']:
            model.train() if phase == 'train' else model.eval()
            running_loss, running_corrects = 0.0, 0

            for inputs, labels in dataloaders[phase]:
                inputs, labels = inputs.to(device), labels.to(device)
                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)

                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

            epoch_loss = running_loss / len(dataloaders[phase].dataset)
            epoch_acc = (running_corrects.double() / len(dataloaders[phase].dataset)).item()

            history[f'{phase}_loss'].append(epoch_loss)
            history[f'{phase}_acc'].append(epoch_acc)

            if phase == 'val' and epoch_acc > best_acc:
                best_acc = epoch_acc
                best_wts = copy.deepcopy(model.state_dict())

    model.load_state_dict(best_wts)
    return model, history

# --- 6. EVALUATION & PLOTTING ---
def plot_history(history, title="Training vs Validation"):
    sns.set_theme(style="whitegrid")
    epochs = range(1, len(history['train_loss']) + 1)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    # Accuracy Plot
    ax1.plot(epochs, history['train_acc'], label='Train Acc', marker='o', linewidth=2.5, color='tab:blue')
    ax1.plot(epochs, history['val_acc'], label='Val Acc', marker='s', linewidth=2.5, color='tab:orange')
    ax1.set_title(f'{title}\nAccuracy over Epochs', fontsize=14, fontweight='bold')
    ax1.set_xlabel('Epochs', fontsize=12)
    ax1.set_ylabel('Accuracy', fontsize=12)
    ax1.legend(loc='lower right', fontsize=11)

    # Loss Plot
    ax2.plot(epochs, history['train_loss'], label='Train Loss', marker='o', linewidth=2.5, color='tab:blue')
    ax2.plot(epochs, history['val_loss'], label='Val Loss', marker='s', linewidth=2.5, color='tab:orange')
    ax2.set_title(f'{title}\nLoss over Epochs', fontsize=14, fontweight='bold')
    ax2.set_xlabel('Epochs', fontsize=12)
    ax2.set_ylabel('Loss', fontsize=12)
    ax2.legend(loc='upper right', fontsize=11)

    plt.tight_layout()
    plt.show()

def evaluate_model(model, test_loader, title="Evaluation"):
    model.eval()
    all_preds, all_labels = [], []

    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    print(f"\n--- Classification Report: {title} ---")
    print(classification_report(all_labels, all_preds, target_names=['Negative', 'Positive']))

    cm = confusion_matrix(all_labels, all_preds)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', annot_kws={"size": 14},
                xticklabels=['Negative', 'Positive'], yticklabels=['Negative', 'Positive'])
    plt.title(f'Confusion Matrix: {title}', fontsize=14, fontweight='bold')
    plt.ylabel('True Label', fontsize=12)
    plt.xlabel('Predicted Label', fontsize=12)
    plt.show()

## 6. Execution
Run the experiments for the chosen scenario and normalization strategy.

In [ ]:
# ==========================================
# --- 7. EXECUTION (RUN YOUR EXPERIMENTS) ---
# ==========================================
import os
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Choose your scenario ('city', 'highway', 'residential', 'overall')
current_scenario = 'city'

# We will now loop through both normalization types to compare them
norm_settings = ['custom', 'conventional']

# Dictionary to store histories for the final comparison graph
all_histories = {}
num_training_epochs = 10

expected_file = f"{ANALYSIS_FOLDER}/image_master_file_{current_scenario}_pedestrian.csv"
if not os.path.exists(expected_file):
    print(f"Error: The file {expected_file} was not found.")
    print("Please make sure you have run the 'Final_DataPreparation.ipynb' notebook to generate the master CSV files before running this training script.")
else:
    for norm_setting in norm_settings:
        print(f"\n{'='*60}")
        print(f"--- RUNNING SCENARIO: {current_scenario.upper()} | NORMALIZATION: {norm_setting.upper()} ---")
        print(f"{'='*60}")

        dataloaders = get_dataloaders(current_scenario, norm_type=norm_setting)
        criterion = nn.CrossEntropyLoss()

        # --- Train & Eval Custom CNN ---
        print(f"\n[Training Custom CNN - {norm_setting} norm]")
        cnn_model = CustomCNN()
        cnn_opt = optim.Adam(cnn_model.parameters(), lr=0.001)
        best_cnn, cnn_history = train_model(cnn_model, dataloaders, criterion, cnn_opt, num_epochs=num_training_epochs)

        plot_history(cnn_history, title=f"Custom CNN ({current_scenario}, {norm_setting})")
        evaluate_model(best_cnn, dataloaders['test'], title=f"Custom CNN ({current_scenario}, {norm_setting})")
        all_histories[f'CustomCNN_{norm_setting}'] = cnn_history

        # --- Train & Eval VGG16 ---
        print(f"\n[Training VGG16 - {norm_setting} norm]")
        vgg_model = get_vgg16()
        vgg_opt = optim.Adam(vgg_model.classifier.parameters(), lr=0.0001)
        best_vgg, vgg_history = train_model(vgg_model, dataloaders, criterion, vgg_opt, num_epochs=num_training_epochs)

        plot_history(vgg_history, title=f"VGG16 ({current_scenario}, {norm_setting})")
        evaluate_model(best_vgg, dataloaders['test'], title=f"VGG16 ({current_scenario}, {norm_setting})")
        all_histories[f'VGG16_{norm_setting}'] = vgg_history

    # ==========================================
    # --- 8. FINAL COMPARISON PLOTS ---
    # ==========================================
    print("\n" + "="*60)
    print("--- FINAL COMPARISON: ALL MODELS & NORMALIZATIONS ---")
    print("="*60)

    sns.set_theme(style="whitegrid")
    epochs = range(1, num_training_epochs + 1)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

    # Distinct colors and markers for clear differentiation
    style_dict = {
        'CustomCNN_custom': {'color': 'tab:blue', 'marker': 'o'},
        'CustomCNN_conventional': {'color': 'tab:cyan', 'marker': 's'},
        'VGG16_custom': {'color': 'tab:red', 'marker': '^'},
        'VGG16_conventional': {'color': 'tab:orange', 'marker': 'D'}
    }

    for name, hist in all_histories.items():
        c = style_dict[name]['color']
        m = style_dict[name]['marker']
        # Validation Accuracy Comparison
        ax1.plot(epochs, hist['val_acc'], label=name, marker=m, linewidth=2.5, color=c)
        # Validation Loss Comparison
        ax2.plot(epochs, hist['val_loss'], label=name, marker=m, linewidth=2.5, color=c)

    ax1.set_title(f'Validation Accuracy Comparison ({current_scenario})', fontsize=15, fontweight='bold')
    ax1.set_xlabel('Epochs', fontsize=13)
    ax1.set_ylabel('Validation Accuracy', fontsize=13)
    ax1.legend(loc='lower right', fontsize=11)

    ax2.set_title(f'Validation Loss Comparison ({current_scenario})', fontsize=15, fontweight='bold')
    ax2.set_xlabel('Epochs', fontsize=13)
    ax2.set_ylabel('Validation Loss', fontsize=13)
    ax2.legend(loc='upper right', fontsize=11)

    plt.tight_layout()
    plt.show()

Loading data for city scenario using custom normalization...
Error: The file /content/drive/MyDrive/Franklin_Project_2024/data/analysis/image_master_file_city_pedestrian.csv was not found.
Please make sure you have run the 'Final_DataPreparation.ipynb' notebook to generate the master CSV files before running this training script.
